In [ ]:
# Загружаем все необходимые библиотеки
import math
import random
import numpy as np
import os
from pathlib import Path

# Практические задания

#### 1. Реализуйте функцию norm(v, p) — Lp-норма вектора (без np.linalg). Проверьте на p=1, 2, ∞.

In [ ]:
# Создаем вектор
x = [3.0, -1.5, 2.7, 0.0, 5.1]

# Создаем функцию
def norm(v, p):
    if not v:  # Проверяем, что вектор не пустой (иначе норма 0)
        return 0.0
    if p == float('inf'):  # Проверяем вектор на бесконечность (возращаем максимальный модуль из вектора)
        return max(abs(i) for i in v)
    else:
        sum_p = 0  # Инициализируем счетчик
        for i in v:  # Проходим по каждому элементу вектора
            sum_p += abs(i) ** p  # Возводим каждый элемент в квадрат
        return sum_p ** (1 / p)  # Возращаем корень суммы квадратов

# Вызов функции        
print(norm(x, 1))  # L1 норма — сумма модулей
print(norm(x, 2))  # L2 норма — длина модулей (евклидова)
print(norm(x, float('inf')))  # L∞-норма - максимум модуля

#### 2. Косинус без np. Реализуйте косинусное расстояние, используя только math.sqrt и циклы. Сравните с NumPy-версией на 10 случайных парах.

In [ ]:
# Создаем вспомогательную функцию скалярного произведения
def dot(a, b):
    if len(a) != len(b):  # Определяем равность векторов
        raise ValueError('Ошибка: Векторы должны иметь одинаковую длину')
    result = 0  # Инициализируем счетчик
    for x, y in zip(a, b):  # Проходим по обоим векторам одновременно
        result += x * y  # Умножаем пару и добавляем в счетчик
    return result  # Возвращаем результат
    
# Создаем вспомогательную функцию для нахождения длины векторов (L2-норма)
def l2_norm(v):
    sum_sq = 0  # Инициализируем счетчик
    for x in v:
        sum_sq += x * x  # Возводим в квадрат и добавляем в счетчик
    return math.sqrt(sum_sq)  # Возвращаем квадратный корень из счетчика
    
# Создаем основную функцию для вычисления расстояния косинуса
def cosine_distance(a, b, verbose=False):
    num = dot(a, b)  # Считаем скалярное произведение
    len_a = l2_norm(a)  #Считаем длину вектора a
    len_b = l2_norm(b)  #Считаем длину вектора b
    den = l2_norm(a) * l2_norm(b)  # Вычисляем знаменатель
    if verbose:  # Отладка (нужно verbose включить True)
        print(f'[DEBAG] dot={num}, |a|={len_a}, |b|={len_b}')
    if den < 1e-12:  # Защита от деления на ноль (1e-12 очень маленькое число)
        return 1.0 # Если вектор нулевой, считаем, что расстояние максимальное (1.0)
    sim = num / den # Считаем косинус (похожесть)
    
    return 1.0 - sim  # Считаем расстояние. Если похожесть 1 (векторы одинаковые), расстояние будет 0.

In [ ]:
# Делаем сравнение с NumPy-версией на 10 случайных парах
print("Начинаем тестирование на 10 случайных векторах...")
print("-" * 50)

# Цикл повторяет действие 10 раз
for i in range(10):
    # Создаём два случайных вектора длиной 5
    a = [random.uniform(-5, 5) for _ in range(5)]
    b = [random.uniform(-5, 5) for _ in range(5)]
    my_result = cosine_distance(a, b)  # Вызываем функцию косинуса (в которой находили расстояние)
    # Превращаем списки в массивы numpy
    a_np = np.array(a)
    b_np = np.array(b)
    # dot(a,b) / (norm(a)*norm(b)) — по аналогии раннее. Мы вычитаем из 1.
    np_sim = np.dot(a_np, b_np) / (np.linalg.norm(a_np) * np.linalg.norm(b_np))
    np_result = 1.0 - np_sim

    # Проверяем, насколько ваши результаты близки.
    # math.isclose возвращает True, если числа почти равны
    is_match = math.isclose(my_result, np_result, rel_tol=1e-5)
    status = 'Совпало' if is_match else 'РАЗЛИЧАЕТСЯ'
    print(f"Тест {i+1}: Мой ответ: {my_result:.4f} | NumPy: {np_result:.4f} | {status}")  # Вывод результатов

#### 3. Топ-5 похожих слов. Загрузите готовые word-эмбеддинги (например, GloVe 50d). Для слова king найдите 5 ближайших по косинусу.

In [ ]:
# Определяем путь к файлу источника
notebook_folder = Path().resolve()
project_root = notebook_folder.parent.parent
file_path = project_root / 'data' / 'glove.6B.50d.txt'

# Построчное чтение файла
embeddings = {}  # Создаем пустой словарь
with open(file_path, 'r', encoding='utf-8') as f:  # Открываем файл
    for line in f:
        values = line.strip().split()  # Разделяем строку на слово
        word = values[0]  # Cохранение слов и их 50-мерных векторов в словарь
        vector = np.array([float(num) for num in values[1:]], dtype='float32')  # Преобразовываем числа в список float
        embeddings[word] = vector  

# Проверка наличия целевого слова
if 'king' not in embeddings:  # Подготовка поиска для king
    print("Ошибка: слово 'king' не найдено в эмбеддингах")  # Защита от KeyError: проверяем наличие слова в загруженных данных
else:
    vector_king = embeddings['king']
    candidates = []  # Создаем пустой список

    # Вычисление сходства со всеми словами
    for word in embeddings:
        if word == 'king':  # Если слово равно king то пропускаем
            continue
        vector_word = embeddings[word]
    
    # Расчёт косинусного сходства с каждым словом, исключая само целевое
    dot_product = np.dot(vector_king, vector_word)  # Вычисляем скалярное произведение векторов
    # Нормы (длины) векторов
    norm_king = np.linalg.norm(vector_king)
    norm_word = np.linalg.norm(vector_word)
    # Находим косинусное сходство
    sim = dot_product / (norm_king * norm_word)
    candidates.append((word, sim))  # Добавляем ключ-сходство в список

    # Сортировка по убыванию и выборка топ-5 результатов
    candidates.sort(key=lambda x: x[1], reverse=True)
    top_5 = candidates[:5]
    print('5 ближайших слов к "king" по косинусному сходству:')
    for word, sim in top_5:
        print(f'{word}: {sim:.4f}')

#### 4. Король − мужчина + женщина = ? Проверьте классическое равенство на эмбеддингах. Должно выйти что-то близкое к queen.

#### 5. Нормализация. Напишите функцию normalize(v), возвращающую единичный вектор того же направления. Что произойдёт, если подать нулевой вектор?

#### 6. Проекция. Реализуйте проекцию вектора a на b: $$\text{proj}_b \mathbf{a} = \frac{\mathbf{a} \cdot \mathbf{b}}{\mathbf{b} \cdot \mathbf{b}} \mathbf{b}$$Визуализируйте на 2D через matplotlib.

#### 7. Угол между документами. Возьмите 5 коротких текстов. Постройте bag-of-words векторы. Посчитайте попарные косинусные близости. Какие документы оказались похожими?

#### 8. Перпендикулярность. Сгенерируйте случайный вектор в R^100. Найдите хотя бы один ненулевой вектор, перпендикулярный ему (скалярное произведение = 0).